# Exercise: Is your house worth it's weight in Dubloons?

You'd need pirate treasure to but a house in London in 2025, so let's find out if you'd be better of burying your money as Dubloons or buying a house with it. (Data restricted to post January 2023 to keep Trino from running out of memory on my ailing M1)

![](pirate.jpg)



In [9]:
import polars as pl
from utils import catalog, engine
from pyiceberg.schema import Schema, NestedField
from pyiceberg.types import DecimalType, DateType, StringType
from pyiceberg.partitioning import PartitionSpec, YearTransform, PartitionField
from IPython.display import display
pl.Config.set_thousands_separator(",")

polars.config.Config

In [13]:
sql = """
-- Convert USD gold prices into GBP denominated prices
with gold_prices as (
    select commodities.gold.date as gold_date, 
           commodities.gold.price * fx.rates.exchange_rate as gold_price
    from commodities.gold
    join fx.rates on fx.rates.date = commodities.gold.date
    where commodities.gold.date >= DATE '2023-01-01'  -- Add date filter if appropriate
), 
filtered_profits as (
    select * from housing.profits 
    where first_day >= DATE '2023-01-01'  -- Match the date filter
),
gold_purchase as (
    select address_id, 
           cast(first_price as DOUBLE) / cast(gold_price as DOUBLE) as purchased_gold,
           first_day,
           last_day,
           cast(gold_price as DOUBLE) as gold_price_at_purchase  -- Cast to DOUBLE
    from filtered_profits
    join gold_prices on gold_date = filtered_profits.first_day
), 
gold_sell as (
    select gp.address_id,
           cast((gp.purchased_gold * cast(gold_prices.gold_price as DOUBLE)) - cast(fp.first_price as DOUBLE) as DOUBLE) as gold_profit,
           cast(fp.profit as DOUBLE) as house_profit,  -- Cast to DOUBLE
           cast(fp.first_price as DOUBLE) as first_price,  -- Cast to DOUBLE
           cast(fp.last_price as DOUBLE) as last_price,   -- Cast to DOUBLE
           gp.gold_price_at_purchase,
           cast(gold_prices.gold_price as DOUBLE) as gold_price_at_sale  -- Cast to DOUBLE
    from gold_purchase gp
    join filtered_profits fp on fp.address_id = gp.address_id
    join gold_prices on gold_prices.gold_date = gp.last_day
)
select * from gold_sell
"""

In [14]:
gold_vs_house_profits = pl.read_database(sql, engine)
gold_vs_house_profits

address_id,gold_profit,house_profit,first_price,last_price,gold_price_at_purchase,gold_price_at_sale
str,f64,f64,f64,f64,f64,f64
"""7E59171D3053F6B6AFA8F7055E9749…","4,472.389114",0.0,"420,000.0","420,000.0","1,575.53076","1,592.307871"
"""7E59171D3053F6B6AFA8F7055E9749…","4,472.389114",0.0,"420,000.0","420,000.0","1,575.53076","1,592.307871"
"""7E59171D3053F6B6AFA8F7055E9749…","4,472.389114",0.0,"420,000.0","420,000.0","1,575.53076","1,592.307871"
"""7E59171D3053F6B6AFA8F7055E9749…","4,472.389114",0.0,"420,000.0","420,000.0","1,575.53076","1,592.307871"
"""7E59171D3053F6B6AFA8F7055E9749…","4,472.389114",0.0,"420,000.0","420,000.0","1,575.53076","1,592.307871"
…,…,…,…,…,…,…
"""523EA4049BE7C657617E05D55E84B6…","10,636.742109","747,275.0","212,725.0","960,000.0","1,536.985928","1,613.83878"
"""523EA4049BE7C657617E05D55E84B6…","10,636.742109","747,275.0","212,725.0","960,000.0","1,536.985928","1,613.83878"
"""523EA4049BE7C657617E05D55E84B6…","10,636.742109","747,275.0","212,725.0","960,000.0","1,536.985928","1,613.83878"


In [16]:
with pl.Config(set_tbl_rows=100):
    # Constants for doubloon calculations
    DOUBLOON_FINE_GOLD_GRAMS = 6.2  # grams of fine gold per doubloon
    TROY_OUNCE_TO_GRAMS = 31.1035   # conversion factor
    DOUBLOON_FINE_GOLD_OZ = DOUBLOON_FINE_GOLD_GRAMS / TROY_OUNCE_TO_GRAMS  # ~0.199 troy oz
    
    summary_df = (
        gold_vs_house_profits
        # .filter(pl.col("county") == "KENT")  # Add this line to filter by county
        .with_columns([
            # Calculate house price in troy ounces of gold at purchase time
            (pl.col("house_profit") + pl.col("first_price")).alias("house_sale_price"),
            
            # Calculate house price in grams of gold (assuming gold price is per troy ounce)
            ((pl.col("house_profit") + pl.col("first_price")) / pl.col("gold_price_at_sale") * TROY_OUNCE_TO_GRAMS).alias("house_price_in_gold_grams"),
            
            # Calculate house price in troy ounces of gold
            ((pl.col("house_profit") + pl.col("first_price")) / pl.col("gold_price_at_sale")).alias("house_price_in_gold_oz"),
            
            # Calculate number of doubloons that could be made from that gold
            ((pl.col("house_profit") + pl.col("first_price")) / pl.col("gold_price_at_sale") * TROY_OUNCE_TO_GRAMS / DOUBLOON_FINE_GOLD_GRAMS).alias("doubloons_from_house_gold")
        ])
        .select([
            "address_id",
            # "county",
            "house_sale_price",
            "house_price_in_gold_grams",
            "house_price_in_gold_oz", 
            "doubloons_from_house_gold",
            # Create quantile bins for doubloon counts
            pl.col("doubloons_from_house_gold")
            .qcut(100, labels=[f"Q{i + 1}" for i in range(100)], include_breaks=True)
            .alias("doubloon_quantiles")
        ])
        .unnest("doubloon_quantiles")
        .unique()
        .sort("breakpoint")
    )
    display(summary_df)

address_id,house_sale_price,house_price_in_gold_grams,house_price_in_gold_oz,doubloons_from_house_gold,breakpoint,category
str,f64,f64,f64,f64,f64,cat
"""AF3B972D3CF6932A73DF3EA37F0860…","20,000.0",389.756775,12.530962,62.863996,124.698051,"""Q1"""
"""6E1F8803A452CA3E0A80A8F0A844CB…","30,000.0",616.865547,19.832673,99.494443,124.698051,"""Q1"""
"""4E5B5654F980EB33077213580DD43B…","35,000.0",689.851918,22.179238,111.266438,124.698051,"""Q1"""
"""5D2E62F21DEEA24D76897A216F9C24…","23,000.0",445.233611,14.314582,71.811873,124.698051,"""Q1"""
"""AB4A430CB01CEBBB487770B99FCDF8…","37,500.0",757.689433,24.360263,122.207973,124.698051,"""Q1"""
"""52C18F9DE05221ED37B58C6023F2FD…","40,000.0",773.127918,24.856621,124.698051,124.698051,"""Q1"""
"""B53973D68E170C8A2EBF7821D41FDA…","20,000.0",385.8157,12.404254,62.228339,124.698051,"""Q1"""
"""FF4A67D4DD1880CBFDE9D626E20ED3…","26,750.0",544.746635,17.513998,87.862361,124.698051,"""Q1"""
"""92FAEF92D0FFBC2B064E20BE6D9E2A…","14,000.0",279.690993,8.992268,45.11145,124.698051,"""Q1"""
